# gesn-search: итерация 2 — исправления по разбору ошибок

Две модели поверх той же e5-base, чтобы разделить эффекты:
- **amb** — синтетика v1, но неоднозначные запросы («мягкая кровля» у норм из разных таблиц) не выбрасываются, а учатся с несколькими правильными ответами;
- **gloss** — новая синтетика: LLM сначала составляет для каждой таблицы норм словарь разговорных названий («побелка», «армопояс»…), затем пишет запросы с их использованием; неоднозначные тоже оставлены.

Словарь строит сама LLM по текстам норм — тестовые запросы в этом не участвуют.

Сравнение на ручном тесте (разметка v2): BM25, e5, ft (итерация 1), amb, gloss.

**Перед запуском:**
- **Add Input** → ноутбук с первым полным прогоном (нужны `e5-gesn.zip` и `synth_raw.jsonl`).
- Session options: **GPU T4 x2**, Internet **On**.
- **Save Version → Save & Run All** (~1,5 часа, вкладку можно закрыть).

In [ ]:
%cd /tmp
!rm -rf /tmp/gesn-search && git clone -q https://github.com/malma0/gesn-search /tmp/gesn-search
%cd /tmp/gesn-search
!pip install -q -e . --no-deps
!pip install -q "rank-bm25>=0.2.2" "pymorphy3>=2.0" "sentence-transformers>=5.0" "accelerate>=1.1.0" datasets

In [ ]:
import glob, shutil, zipfile
from pathlib import Path

def find(name):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert found, f"Не найден {name} — добавь вывод первого полного прогона через Add Input"
    return found[0]

zipfile.ZipFile(find("e5-gesn.zip")).extractall("models/e5-gesn-v1")
Path("data/train").mkdir(parents=True, exist_ok=True)
shutil.copy(find("synth_raw.jsonl"), "data/train/synth_raw_v1.jsonl")
print("ok")

## Быстрая проверка нового кода генерации (8 норм)

In [ ]:
!python -m gesn.generate --glossary /tmp/check_glossary.jsonl --out /tmp/check_raw.jsonl --limit 8
!head -c 1200 /tmp/check_glossary.jsonl; echo; head -c 800 /tmp/check_raw.jsonl

## B. Словарь разговорных названий и новая синтетика

In [ ]:
!python -m gesn.generate --glossary data/train/glossary.jsonl --out data/train/synth_raw_gloss.jsonl

In [ ]:
import json
for line in list(open("data/train/glossary.jsonl", encoding="utf-8"))[:15]:
    r = json.loads(line)
    print(r["table_code"], "→", ", ".join(r["items"][:8]))

## A. Синтетика v1 с неоднозначными запросами

In [ ]:
!python -m gesn.synth --raw data/train/synth_raw_v1.jsonl --keep-ambiguous --out-dir data/train/amb
!CUDA_VISIBLE_DEVICES=0 python -m gesn.train --train data/train/amb/train.jsonl --out models/e5-gesn-amb

## B. Обучение на синтетике со словарём

In [ ]:
!python -m gesn.synth --raw data/train/synth_raw_gloss.jsonl --keep-ambiguous --out-dir data/train/gloss
!CUDA_VISIBLE_DEVICES=0 python -m gesn.train --train data/train/gloss/train.jsonl --out models/e5-gesn-gloss

## Оценка на ручном тесте (разметка v2)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.evaluate --methods bm25 e5 ft=models/e5-gesn-v1 amb=models/e5-gesn-amb gloss=models/e5-gesn-gloss bm25+gloss --out results/iter2_test.md

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m gesn.analyze --methods e5 ft=models/e5-gesn-v1 amb=models/e5-gesn-amb gloss=models/e5-gesn-gloss --out-dir results/iter2

## Сохранение

In [ ]:
out = Path("/kaggle/working")
for f in ["results/iter2_test.md", "results/iter2/significance.md", "results/iter2/per_query.jsonl",
          "data/train/glossary.jsonl", "data/train/synth_raw_gloss.jsonl"]:
    shutil.copy(f, out / Path(f).name)
for m in ["amb", "gloss"]:
    shutil.make_archive(str(out / f"e5-gesn-{m}"), "zip", f"models/e5-gesn-{m}")
print(sorted(p.name for p in out.iterdir()))

In [ ]:
!cat results/iter2/per_query.jsonl